# Cluster AgentForge SWE-smith Run

Use this notebook on an interactive cluster node to preview, collect, evaluate, and inspect SWE-smith trajectories. It uses the repository scripts and pinned dataset/dependency configuration rather than reimplementing the pipeline in notebook cells.

All expensive or state-changing actions are disabled by default. Enable only the relevant `RUN_*` or `SUBMIT_*` switch after reviewing its preview.

## 0. Repository and scratch setup

The selected kernel should be the `debug-depo` environment created by `cluster/setup_jupyter_env.sh` or another environment prepared with `uv sync` plus the two installer scripts.

In [ ]:
from datetime import datetime
from pathlib import Path
import glob
import json
import os
import shlex
import shutil
import subprocess
import sys


def find_repo_root():
    candidates = []
    for candidate in [
        os.getenv('DEBUG_DEPO_ROOT'),
        Path.cwd(),
        Path.cwd() / 'debug-depo',
        Path.cwd().parent,
        Path.home() / 'debug-depo',
    ]:
        if not candidate:
            continue
        path = Path(candidate).expanduser().resolve()
        if path not in candidates:
            candidates.append(path)
    for candidate in candidates:
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    searched = '\n'.join(f'  - {candidate}' for candidate in candidates)
    raise FileNotFoundError(f'Could not find debug-depo. Searched:\n{searched}')


def find_ephemeral_root(root):
    candidates = [
        os.getenv('DEBUG_DEPO_EPHEMERAL'),
        Path(os.environ['RDS']) / 'ephemeral' / 'debug-depo' if os.getenv('RDS') else None,
        Path(os.environ['EPHEMERAL']) / 'debug-depo' if os.getenv('EPHEMERAL') else None,
        Path(os.environ['SCRATCH']) / 'debug-depo' if os.getenv('SCRATCH') else None,
        root / 'scratch',
    ]
    for candidate in candidates:
        if not candidate:
            continue
        path = Path(candidate).expanduser().resolve()
        if path.exists() or path.parent.exists():
            return path
    return (root / 'scratch').resolve()


ROOT = find_repo_root()
kernel_bin = Path(sys.executable).resolve().parent
ephemeral = find_ephemeral_root(ROOT)
scratch = Path(os.getenv('DEBUG_DEPO_SCRATCH') or ephemeral).expanduser().resolve()

env = os.environ.copy()
env['PATH'] = str(kernel_bin) + os.pathsep + env.get('PATH', '')
python_paths = [ROOT / 'src', ROOT / 'external' / 'mini-swe-agent-plus' / 'src', ROOT / 'external' / 'SWE-smith']
env['PYTHONPATH'] = os.pathsep.join(str(path) for path in python_paths) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
env.setdefault('DEBUG_DEPO_ROOT', str(ROOT))
env.setdefault('DEBUG_DEPO_EPHEMERAL', str(ephemeral))
env.setdefault('DEBUG_DEPO_SCRATCH', str(scratch))
env.setdefault('HF_HOME', str(scratch / 'huggingface'))
env.setdefault('UV_CACHE_DIR', str(scratch / 'uv-cache'))
env.setdefault('TMPDIR', str(scratch / 'tmp'))
env.setdefault('APPTAINER_CACHEDIR', str(scratch / 'apptainer-cache'))
env.setdefault('VLLM_IMAGE', str(ROOT / 'cluster' / 'apptainer' / 'vllm-openai.sif'))
env.setdefault('SWESMITH_APPTAINER_CACHE_DIR', str(scratch / 'swesmith_cache' / 'apptainer-cache'))
env.setdefault('SWESMITH_APPTAINER_SIF_DIR', str(scratch / 'swesmith_cache' / 'sifs'))
kernel_uv = kernel_bin / 'uv'
if kernel_uv.exists():
    env.setdefault('UV', str(kernel_uv))

for path in [
    scratch,
    Path(env['HF_HOME']),
    Path(env['UV_CACHE_DIR']),
    Path(env['TMPDIR']),
    Path(env['APPTAINER_CACHEDIR']),
    Path(env['SWESMITH_APPTAINER_CACHE_DIR']),
    Path(env['SWESMITH_APPTAINER_SIF_DIR']),
    Path(env['VLLM_IMAGE']).parent,
]:
    path.mkdir(parents=True, exist_ok=True)

print(f'Repository:     {ROOT}')
print(f'Kernel Python:  {sys.executable}')
print(f'Ephemeral root: {ephemeral}')
print(f'Scratch:        {scratch}')
print(f"TMPDIR:         {env['TMPDIR']}")
print(f"SWE-smith SIFs: {env['SWESMITH_APPTAINER_SIF_DIR']}")

## 1. Run configuration

The mode-dependent defaults match the tracked smoke, 30-task pilot, and 5,000-task full collection workflows. Full mode selects `data/splits/swesmith_train_5000_instance_ids.txt` and uses 25 collection shards. Use a new dated `RUN_NAME` for every submission. The supported workflow is to delete and recreate a run root, not update a historical run in place; overwrite remains disabled by default.

In [ ]:
PBS_MODE = os.getenv('SWESMITH_MODE', 'pilot')
MODE_DEFAULTS = {
    'smoke': {'task_limit': '2', 'expected_tasks': '2', 'num_shards': '1', 'rollout_workers': '2', 'eval_workers': '2'},
    'pilot': {'task_limit': '30', 'expected_tasks': '30', 'num_shards': '3', 'rollout_workers': '5', 'eval_workers': '12'},
    'full': {'task_limit': '', 'expected_tasks': '5000', 'num_shards': '25', 'rollout_workers': '8', 'eval_workers': '25'},
}
if PBS_MODE not in MODE_DEFAULTS:
    raise ValueError(f'SWESMITH_MODE must be smoke, pilot, or full, got: {PBS_MODE}')
mode_defaults = MODE_DEFAULTS[PBS_MODE]

DATASET = os.getenv('DATASET', 'SWE-bench/SWE-smith-py')
DATASET_REVISION = os.getenv('SWESMITH_DATASET_REVISION', '77cab9055d42ab4a5c25c89a8f937096db13558e')
SPLIT = os.getenv('SPLIT', 'train')
TASK_IDS_FILE = os.getenv('TASK_IDS_FILE', 'data/splits/swesmith_train_5000_instance_ids.txt' if PBS_MODE == 'full' else '')

AGENTFORGE_MODEL = os.getenv('AGENTFORGE_MODEL', 'Kwai-Klear/Klear-AgentForge-8B-SFT')
MINI_SWE_MODEL = os.getenv('MINI_SWE_MODEL', f'hosted_vllm/{AGENTFORGE_MODEL}')
MINI_SWE_CONFIG = os.getenv('MINI_SWE_CONFIG', '')
MINI_SWE_RUNNER = os.getenv('MINI_SWE_RUNNER', 'singularity')
MINI_SWE_ENVIRONMENT_CLASS = os.getenv('MINI_SWE_ENVIRONMENT_CLASS', 'singularity')
MSWEA_SINGULARITY_EXECUTABLE = os.getenv('MSWEA_SINGULARITY_EXECUTABLE', 'apptainer')
LLM_BASE_URL = os.getenv('LLM_BASE_URL', 'http://127.0.0.1:8000/v1').rstrip('/')
LLM_API_KEY = os.getenv('LLM_API_KEY', 'local')

RUN_NAME = os.getenv('RUN_NAME', f'swesmith-{PBS_MODE}-{datetime.now():%Y%m%d}')
RUN_ROOT = Path(os.getenv('RUN_ROOT', scratch / 'runs' / RUN_NAME)).expanduser().resolve()
CLUSTER_LOG_DIR = Path(os.getenv('CLUSTER_LOG_DIR', RUN_ROOT / 'cluster-logs')).expanduser().resolve()
TEMPERATURES = os.getenv('TEMPERATURES', '0.6:0.7')
RUNS_PER_TEMPERATURE = int(os.getenv('RUNS_PER_TEMPERATURE', '4'))
TOTAL_SAMPLES = len([value for value in TEMPERATURES.split(':') if value]) * RUNS_PER_TEMPERATURE
BASE_SEED = int(os.getenv('BASE_SEED', '42'))
task_limit_value = os.getenv('TASK_LIMIT', mode_defaults['task_limit'])
TASK_LIMIT = int(task_limit_value) if task_limit_value else None
expected_tasks_default = str(TASK_LIMIT) if TASK_LIMIT is not None else mode_defaults['expected_tasks']
EXPECTED_TASKS = int(os.getenv('EXPECTED_TASKS', expected_tasks_default))
NUM_SHARDS = int(os.getenv('NUM_SHARDS', mode_defaults['num_shards']))
SHARD_INDEX = int(os.getenv('SHARD_INDEX', os.getenv('PBS_ARRAY_INDEX', '0')))

MAX_STEPS = int(os.getenv('MAX_STEPS', '200'))
CONTEXT_LENGTH = int(os.getenv('CONTEXT_LENGTH', '65536'))
TOP_P = float(os.getenv('TOP_P', '1.0'))
TIMEOUT_SECONDS = int(os.getenv('TIMEOUT_SECONDS', '21600'))
ROLLOUT_WORKERS = int(os.getenv('ROLLOUT_WORKERS', mode_defaults['rollout_workers']))
MINI_SWE_WORKERS = int(os.getenv('MINI_SWE_WORKERS', '1'))
EVAL_MAX_WORKERS = int(os.getenv('EVAL_MAX_WORKERS', mode_defaults['eval_workers']))
EVAL_TIMEOUT = os.getenv('EVAL_TIMEOUT', '')
SWESMITH_EVAL_RUNTIME = os.getenv('SWESMITH_EVAL_RUNTIME', 'apptainer')
SUBMIT_EVAL = os.getenv('SUBMIT_EVAL', '0' if PBS_MODE == 'full' else '1') == '1'
SUBMIT_ANALYSIS = os.getenv('SUBMIT_ANALYSIS', '1' if SUBMIT_EVAL else '0') == '1'
ALLOW_MONOLITHIC_FULL_EVAL = os.getenv('ALLOW_MONOLITHIC_FULL_EVAL', '0') == '1'

preview = {
    'mode': PBS_MODE,
    'dataset': DATASET,
    'dataset_revision': DATASET_REVISION,
    'split': SPLIT,
    'model': AGENTFORGE_MODEL,
    'mini_model': MINI_SWE_MODEL,
    'mini_runner': MINI_SWE_RUNNER,
    'mini_environment_class': MINI_SWE_ENVIRONMENT_CLASS,
    'llm_base_url': LLM_BASE_URL,
    'run_root': str(RUN_ROOT),
    'cluster_log_dir': str(CLUSTER_LOG_DIR),
    'task_ids_file': TASK_IDS_FILE or '<dataset selection>',
    'task_limit': TASK_LIMIT if TASK_LIMIT is not None else '<all selected tasks>',
    'temperatures': TEMPERATURES,
    'runs_per_temperature': RUNS_PER_TEMPERATURE,
    'total_samples_per_task': TOTAL_SAMPLES,
    'expected_tasks': EXPECTED_TASKS,
    'num_shards': NUM_SHARDS,
    'shard_index': SHARD_INDEX,
    'rollout_workers': ROLLOUT_WORKERS,
    'eval_workers': EVAL_MAX_WORKERS,
    'submit_eval': SUBMIT_EVAL,
    'submit_analysis': SUBMIT_ANALYSIS,
}
print(json.dumps(preview, indent=2))

if MINI_SWE_RUNNER == 'pool_way':
    raise ValueError('SWE-smith cannot use pool_way because it ignores task-branch startup commands.')
if TOTAL_SAMPLES < 1 or NUM_SHARDS < 1 or EXPECTED_TASKS < 1:
    raise ValueError('TOTAL_SAMPLES, NUM_SHARDS, and EXPECTED_TASKS must be positive.')
if NUM_SHARDS > EXPECTED_TASKS:
    raise ValueError('NUM_SHARDS cannot exceed EXPECTED_TASKS.')
if not 0 <= SHARD_INDEX < NUM_SHARDS:
    raise ValueError(f'Invalid shard {SHARD_INDEX} for NUM_SHARDS={NUM_SHARDS}')
if SUBMIT_ANALYSIS and not SUBMIT_EVAL:
    raise ValueError('SUBMIT_ANALYSIS requires SUBMIT_EVAL.')
if RUN_ROOT.exists() and any(RUN_ROOT.iterdir()):
    print(f'WARNING: {RUN_ROOT} is not empty. Delete it or choose a new dated RUN_NAME before submitting.')

## 2. Environment preflight

This checks the selected Python environment and cluster executables without starting containers or model inference.

In [ ]:
python_check = [
    sys.executable,
    '-c',
    'import debug_depo, minisweagent, swesmith; print("Python packages: OK")',
]
print(shlex.join(python_check))
subprocess.run(python_check, cwd=ROOT, env=env, check=True)

for executable in ['uv', MSWEA_SINGULARITY_EXECUTABLE, 'qsub']:
    resolved = shutil.which(executable, path=env['PATH'])
    print(f'{executable}: {resolved or "<not found>"}')

for required in [
    ROOT / 'scripts' / 'collect_swesmith.sh',
    ROOT / 'scripts' / 'evaluate_swesmith.sh',
    ROOT / 'scripts' / 'analyze_swesmith.sh',
    ROOT / 'cluster' / 'submit_swesmith.sh',
]:
    if not required.is_file():
        raise FileNotFoundError(required)
print('Repository scripts: OK')

## 3. Optional vLLM launch and health check

Leave `START_VLLM` off if the model server is managed by another job. The model check sends a tiny completion only when `RUN_MODEL_CHECK` is enabled.

In [ ]:
START_VLLM = False
RUN_MODEL_CHECK = False
vllm_process = None

if START_VLLM:
    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    server_env = env.copy()
    server_env.update({
        'AGENTFORGE_MODEL': AGENTFORGE_MODEL,
        'VLLM_MODEL': AGENTFORGE_MODEL,
        'MINI_SWE_MODEL': MINI_SWE_MODEL,
        'LLM_BASE_URL': LLM_BASE_URL,
        'LLM_API_KEY': LLM_API_KEY,
        'CONTEXT_LENGTH': str(CONTEXT_LENGTH),
        'MSWEA_SINGULARITY_EXECUTABLE': MSWEA_SINGULARITY_EXECUTABLE,
    })
    log_path = RUN_ROOT / 'vllm.log'
    log_handle = log_path.open('w', encoding='utf-8')
    vllm_process = subprocess.Popen(
        ['bash', 'cluster/apptainer/serve_vllm.sh'],
        cwd=ROOT,
        env=server_env,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
    )
    (RUN_ROOT / 'vllm.pid').write_text(str(vllm_process.pid), encoding='utf-8')
    print(f'Started vLLM pid={vllm_process.pid}; log={log_path}')
else:
    print('START_VLLM is off; expecting an existing server for real collection.')

if RUN_MODEL_CHECK:
    check_cmd = [
        sys.executable,
        '-m',
        'debug_depo.check_local_llm',
        '--base-url', LLM_BASE_URL,
        '--api-key', LLM_API_KEY,
        '--model', AGENTFORGE_MODEL,
        '--timeout', os.getenv('LLM_CHECK_TIMEOUT', '900'),
        '--max-tokens', '16',
    ]
    print(shlex.join(check_cmd))
    subprocess.run(check_cmd, cwd=ROOT, env=env, check=True)
else:
    print('Set RUN_MODEL_CHECK = True before a real rollout.')

## 4. Preview or submit the tracked PBS chain

The dry preview shows the jobs enabled for the selected mode. The notebook keeps full mode collection-only so collection, evaluation, and analysis can be submitted and inspected separately; the tracked full evaluator uses 25 workers, 256 GB, and a 48-hour walltime while processing all 8 sample slots sequentially. Actual submission requires `qsub` and remains disabled until `SUBMIT_PBS_CHAIN` is set.

For the default pilot, the cache-first helper submits `cache → collect → evaluate → analyse` and pins the 30 tasks to `data/splits/swesmith_train_5000_instance_ids.txt`:

```bash
DRY_RUN=1 cluster/submit_swesmith_pilot_with_cache.sh
cluster/submit_swesmith_pilot_with_cache.sh
```

In [ ]:
PREVIEW_PBS_CHAIN = True
SUBMIT_PBS_CHAIN = False

submit_env = env.copy()
submit_env.update({
    'SWESMITH_MODE': PBS_MODE,
    'RUN_NAME': RUN_NAME,
    'RUN_ROOT': str(RUN_ROOT),
    'CLUSTER_LOG_DIR': str(CLUSTER_LOG_DIR),
    'DATASET': DATASET,
    'SWESMITH_DATASET_REVISION': DATASET_REVISION,
    'SPLIT': SPLIT,
    'EXPECTED_TASKS': str(EXPECTED_TASKS),
    'NUM_SHARDS': str(NUM_SHARDS),
    'RUNS_PER_TEMPERATURE': str(RUNS_PER_TEMPERATURE),
    'TEMPERATURES': TEMPERATURES,
    'BASE_SEED': str(BASE_SEED),
    'ROLLOUT_WORKERS': str(ROLLOUT_WORKERS),
    'AGENTFORGE_MODEL': AGENTFORGE_MODEL,
    'MINI_SWE_MODEL': MINI_SWE_MODEL,
    'MINI_SWE_RUNNER': MINI_SWE_RUNNER,
    'MINI_SWE_ENVIRONMENT_CLASS': MINI_SWE_ENVIRONMENT_CLASS,
    'MAX_STEPS': str(MAX_STEPS),
    'CONTEXT_LENGTH': str(CONTEXT_LENGTH),
    'TOP_P': str(TOP_P),
    'TIMEOUT_SECONDS': str(TIMEOUT_SECONDS),
    'EVAL_MAX_WORKERS': str(EVAL_MAX_WORKERS),
    'SWESMITH_EVAL_RUNTIME': SWESMITH_EVAL_RUNTIME,
    'SUBMIT_EVAL': '1' if SUBMIT_EVAL else '0',
    'SUBMIT_ANALYSIS': '1' if SUBMIT_ANALYSIS else '0',
    'OVERWRITE': '0',
})
if TASK_LIMIT is not None:
    submit_env['TASK_LIMIT'] = str(TASK_LIMIT)
if TASK_IDS_FILE:
    submit_env['TASK_IDS_FILE'] = TASK_IDS_FILE
if MINI_SWE_CONFIG:
    submit_env['MINI_SWE_CONFIG'] = MINI_SWE_CONFIG
if EVAL_TIMEOUT:
    submit_env['EVAL_TIMEOUT'] = EVAL_TIMEOUT

if PREVIEW_PBS_CHAIN:
    preview_env = submit_env.copy()
    preview_env['DRY_RUN'] = '1'
    subprocess.run(['bash', 'cluster/submit_swesmith.sh'], cwd=ROOT, env=preview_env, check=True)

if SUBMIT_PBS_CHAIN:
    expected_run_root = (scratch / 'runs' / RUN_NAME).resolve()
    if RUN_ROOT != expected_run_root:
        raise ValueError(f'PBS scripts use {expected_run_root}; set RUN_ROOT to that path before submission.')
    if RUN_ROOT.exists() and any(RUN_ROOT.iterdir()):
        raise FileExistsError(f'Delete the existing run root before submission: {RUN_ROOT}')
    if PBS_MODE == 'full' and SUBMIT_EVAL and not ALLOW_MONOLITHIC_FULL_EVAL:
        raise RuntimeError('Full evaluation is a single 8-sample job. Keep SUBMIT_EVAL=0 or set ALLOW_MONOLITHIC_FULL_EVAL=1 after reviewing the walltime risk.')
    if not PREVIEW_PBS_CHAIN:
        raise RuntimeError('Preview the PBS chain before submission.')
    subprocess.run(['bash', 'cluster/submit_swesmith.sh'], cwd=ROOT, env=submit_env, check=True)
else:
    print('SUBMIT_PBS_CHAIN is off; no jobs submitted.')

## 5. Interactive one-task smoke

This runs one task and one sample slot, rather than the full 8-rollout schedule. It validates model access, the task-branch checkout, the mini-swe Singularity environment, trajectory capture, and prediction writing.

In [ ]:
def make_collection_env(
    run_root,
    *,
    temperatures,
    runs_per_temperature,
    expected_tasks,
    task_limit,
    num_shards,
    shard_index,
    overwrite=False,
):
    output_dir = Path(run_root) / 'collection' / f'shard-{shard_index}'
    collection_env = env.copy()
    for key in ['LIMIT', 'TASK_IDS_FILE', 'INSTANCE_ID', 'START_INDEX', 'OVERWRITE']:
        collection_env.pop(key, None)
    collection_env.update({
        'DATASET': DATASET,
        'SWESMITH_DATASET_REVISION': DATASET_REVISION,
        'SPLIT': SPLIT,
        'OUTPUT_DIR': str(output_dir),
        'EXPECTED_TASKS': str(expected_tasks),
        'NUM_SHARDS': str(num_shards),
        'SHARD_INDEX': str(shard_index),
        'TEMPERATURES': temperatures,
        'RUNS_PER_TEMPERATURE': str(runs_per_temperature),
        'BASE_SEED': str(BASE_SEED),
        'AGENTFORGE_MODEL': AGENTFORGE_MODEL,
        'MINI_SWE_MODEL': MINI_SWE_MODEL,
        'MINI_SWE_RUNNER': MINI_SWE_RUNNER,
        'MINI_SWE_ENVIRONMENT_CLASS': MINI_SWE_ENVIRONMENT_CLASS,
        'MSWEA_SINGULARITY_EXECUTABLE': MSWEA_SINGULARITY_EXECUTABLE,
        'LLM_BASE_URL': LLM_BASE_URL,
        'LLM_API_KEY': LLM_API_KEY,
        'MAX_STEPS': str(MAX_STEPS),
        'CONTEXT_LENGTH': str(CONTEXT_LENGTH),
        'TOP_P': str(TOP_P),
        'TIMEOUT_SECONDS': str(TIMEOUT_SECONDS),
        'ROLLOUT_WORKERS': str(ROLLOUT_WORKERS),
        'MINI_SWE_WORKERS': str(MINI_SWE_WORKERS),
        'STREAM_OUTPUT': os.getenv('STREAM_OUTPUT', '0'),
    })
    if task_limit is not None:
        collection_env['LIMIT'] = str(task_limit)
    if TASK_IDS_FILE:
        collection_env['TASK_IDS_FILE'] = TASK_IDS_FILE
    if MINI_SWE_CONFIG:
        collection_env['MINI_SWE_CONFIG'] = MINI_SWE_CONFIG
    if overwrite:
        collection_env['OVERWRITE'] = '1'
    return output_dir, collection_env


RUN_ONE_TASK_SMOKE = False
SMOKE_RUN_ROOT = Path(os.getenv('SMOKE_RUN_ROOT', scratch / 'runs' / f'swesmith-notebook-smoke-{datetime.now():%Y%m%d}'))
smoke_output, smoke_env = make_collection_env(
    SMOKE_RUN_ROOT,
    temperatures='0.6',
    runs_per_temperature=1,
    expected_tasks=1,
    task_limit=1,
    num_shards=1,
    shard_index=0,
    overwrite=False,
)
print(json.dumps({
    'run': RUN_ONE_TASK_SMOKE,
    'output_dir': str(smoke_output),
    'temperatures': smoke_env['TEMPERATURES'],
    'runs_per_temperature': smoke_env['RUNS_PER_TEMPERATURE'],
}, indent=2))

if RUN_ONE_TASK_SMOKE:
    subprocess.run(['bash', 'scripts/collect_swesmith.sh'], cwd=ROOT, env=smoke_env, check=True)
    summary = json.loads((smoke_output / 'summary.json').read_text(encoding='utf-8'))
    print(json.dumps({key: summary.get(key) for key in ['n_tasks', 'n_rollouts', 'n_finished', 'n_errors', 'n_with_patch']}, indent=2))

## 6. Run one configured shard interactively

Keep this disabled until the one-task smoke succeeds. The selector applies the task limit before deterministic modulo sharding, matching the PBS workflow.

In [ ]:
RUN_CONFIGURED_SHARD = False
OVERWRITE_SHARD = False

shard_output, shard_env = make_collection_env(
    RUN_ROOT,
    temperatures=TEMPERATURES,
    runs_per_temperature=RUNS_PER_TEMPERATURE,
    expected_tasks=EXPECTED_TASKS,
    task_limit=TASK_LIMIT,
    num_shards=NUM_SHARDS,
    shard_index=SHARD_INDEX,
    overwrite=OVERWRITE_SHARD,
)
print(json.dumps({
    'run': RUN_CONFIGURED_SHARD,
    'output_dir': str(shard_output),
    'num_shards': NUM_SHARDS,
    'shard_index': SHARD_INDEX,
    'expected_selected_tasks_before_sharding': EXPECTED_TASKS,
    'total_samples_per_task': TOTAL_SAMPLES,
    'overwrite': OVERWRITE_SHARD,
}, indent=2))

if RUN_CONFIGURED_SHARD:
    subprocess.run(['bash', 'scripts/collect_swesmith.sh'], cwd=ROOT, env=shard_env, check=True)
    print((shard_output / 'summary.json').read_text(encoding='utf-8'))
else:
    print('Dry preview only. Set RUN_CONFIGURED_SHARD = True to collect this shard.')

## 7. Merge and evaluate one sample slot

Evaluation is performed separately for every sample index. The helper preserves prediction metadata such as temperature, seed, and the reverse-application marker used only by SWE-smith gold mocks.

In [ ]:
def sample_prediction_paths(run_root, sample_index):
    return sorted(Path(path) for path in glob.glob(str(Path(run_root) / 'collection' / 'shard-*' / 'samples' / f'sample-{sample_index}' / 'predictions.jsonl')))


def merge_sample(run_root, sample_index):
    paths = sample_prediction_paths(run_root, sample_index)
    if not paths:
        raise FileNotFoundError(f'No shard predictions found for sample {sample_index}')
    merged_dir = Path(run_root) / 'merged' / f'sample-{sample_index}'
    merged_dir.mkdir(parents=True, exist_ok=True)
    merge_env = env.copy()
    merge_env.update({
        'OUTPUT': str(merged_dir / 'predictions.jsonl'),
        'SUMMARY_OUTPUT': str(merged_dir / 'predictions_summary.json'),
    })
    subprocess.run(['bash', 'scripts/merge_predictions.sh', *(str(path) for path in paths)], cwd=ROOT, env=merge_env, check=True)
    return merged_dir / 'predictions.jsonl'


def evaluate_sample(run_root, sample_index, predictions_path, *, overwrite=False, dry_run=False):
    evaluation_dir = Path(run_root) / 'evaluation' / f'sample-{sample_index}'
    evaluation_dir.mkdir(parents=True, exist_ok=True)
    eval_env = env.copy()
    eval_env.update({
        'DATASET': DATASET,
        'SWESMITH_DATASET_REVISION': DATASET_REVISION,
        'SPLIT': SPLIT,
        'PREDICTIONS_PATH': str(predictions_path),
        'SUMMARY_OUTPUT': str(evaluation_dir / 'summary.json'),
        'LOG_DIR': str(evaluation_dir / 'logs'),
        'SWESMITH_EVAL_RUNTIME': SWESMITH_EVAL_RUNTIME,
        'SWESMITH_APPTAINER_CACHE_DIR': env['SWESMITH_APPTAINER_CACHE_DIR'],
        'SWESMITH_APPTAINER_SIF_DIR': env['SWESMITH_APPTAINER_SIF_DIR'],
        'EVAL_MAX_WORKERS': str(EVAL_MAX_WORKERS),
    })
    if TASK_IDS_FILE:
        eval_env['TASK_IDS_FILE'] = TASK_IDS_FILE
    if EVAL_TIMEOUT:
        eval_env['EVAL_TIMEOUT'] = EVAL_TIMEOUT
    if overwrite:
        eval_env['OVERWRITE'] = '1'
    if dry_run:
        eval_env['DRY_RUN'] = '1'
    subprocess.run(['bash', 'scripts/evaluate_swesmith.sh'], cwd=ROOT, env=eval_env, check=True)
    return evaluation_dir / 'summary.json'


SAMPLE_INDEX = 0
RUN_SAMPLE_MERGE = False
RUN_SAMPLE_EVAL = False
DRY_RUN_SAMPLE_EVAL = False
sample_paths = sample_prediction_paths(RUN_ROOT, SAMPLE_INDEX)
print(f'Sample {SAMPLE_INDEX}: {len(sample_paths)} shard prediction files')
for path in sample_paths:
    print(path)

merged_predictions = RUN_ROOT / 'merged' / f'sample-{SAMPLE_INDEX}' / 'predictions.jsonl'
if RUN_SAMPLE_MERGE:
    merged_predictions = merge_sample(RUN_ROOT, SAMPLE_INDEX)
    print(f'Merged predictions: {merged_predictions}')

if RUN_SAMPLE_EVAL:
    if not merged_predictions.is_file():
        raise FileNotFoundError('Merge the selected sample before evaluation.')
    summary_path = evaluate_sample(
        RUN_ROOT,
        SAMPLE_INDEX,
        merged_predictions,
        overwrite=False,
        dry_run=DRY_RUN_SAMPLE_EVAL,
    )
    print(summary_path.read_text(encoding='utf-8'))
else:
    print('Set RUN_SAMPLE_MERGE and RUN_SAMPLE_EVAL explicitly when ready.')

## 8. Analyze the completed run

Run analysis only after every configured sample has merged predictions and an evaluation summary. It writes rollout-level CSV, task-level CSV, per-temperature pass@k, and explicitly labelled mixed-temperature pass@k.

In [ ]:
RUN_ANALYSIS = False
analysis_env = env.copy()
analysis_env.update({
    'RUN_ROOT': str(RUN_ROOT),
    'RUNS_PER_TEMPERATURE': str(RUNS_PER_TEMPERATURE),
    'TOTAL_SAMPLES': str(TOTAL_SAMPLES),
    'EXPECTED_TASKS': str(EXPECTED_TASKS),
})

if RUN_ANALYSIS:
    subprocess.run(['bash', 'scripts/analyze_swesmith.sh'], cwd=ROOT, env=analysis_env, check=True)

analysis_summary_path = RUN_ROOT / 'analysis' / 'summary.json'
if analysis_summary_path.is_file():
    analysis_summary = json.loads(analysis_summary_path.read_text(encoding='utf-8'))
    print(json.dumps({
        'tasks': analysis_summary.get('tasks'),
        'rollouts': analysis_summary.get('rollouts'),
        'evaluated_rollouts': analysis_summary.get('evaluated_rollouts'),
        'resolved_rollouts': analysis_summary.get('resolved_rollouts'),
        'fully_evaluated_tasks': analysis_summary.get('fully_evaluated_tasks'),
        'temperatures': analysis_summary.get('temperatures'),
        'mixed_temperature_pass_at_k': analysis_summary.get('mixed_temperature_pass_at_k'),
    }, indent=2))
else:
    print(f'No analysis summary yet: {analysis_summary_path}')

## 9. Inspect artifacts and failures

This lightweight view works without pandas. For detailed task/trajectory inspection, open `notebooks/inspect_swesmith_collection.ipynb` and point it at the same `RUN_ROOT`.

In [ ]:
manifest_paths = sorted(RUN_ROOT.glob('collection/shard-*/collection_manifest.json'))
sample_summary_paths = sorted(RUN_ROOT.glob('collection/shard-*/samples/sample-*/summary.json'))
evaluation_summary_paths = sorted(RUN_ROOT.glob('evaluation/sample-*/summary.json'))

collection_errors = 0
finished_rollouts = 0
for path in sample_summary_paths:
    payload = json.loads(path.read_text(encoding='utf-8'))
    collection_errors += int(payload.get('n_errors', 0))
    finished_rollouts += int(payload.get('n_finished', payload.get('n_completed', 0)))

evaluation_status_counts = {}
for path in evaluation_summary_paths:
    payload = json.loads(path.read_text(encoding='utf-8'))
    for status, instance_ids in payload.get('status_ids', {}).items():
        evaluation_status_counts[status] = evaluation_status_counts.get(status, 0) + len(instance_ids)

print(json.dumps({
    'run_root': str(RUN_ROOT),
    'collection_manifests': len(manifest_paths),
    'sample_summaries': len(sample_summary_paths),
    'finished_rollouts': finished_rollouts,
    'collection_errors': collection_errors,
    'evaluation_summaries': len(evaluation_summary_paths),
    'evaluation_status_counts': evaluation_status_counts,
}, indent=2))

## Runtime sizing

- Pilot: `30 tasks × 8 samples = 240 trajectories` before retries.
- Full tracked sample: `5,000 × 8 = 40,000 trajectories` across 25 collection shards, or 200 tasks and 1,600 trajectories per shard.
- Full evaluation loops over all 8 sample slots in one 32-CPU, 256 GB, 48-hour PBS job with 25 workers. The notebook defaults full mode to `SUBMIT_EVAL=0` so that stage can be submitted separately after collection is inspected.
- `ROLLOUT_WORKERS` controls concurrent task/sample subprocesses inside one shard. `MINI_SWE_WORKERS` normally remains one because each adapter invocation contains one task.
- SWE-smith images are repository-level, so different tasks may share a cached SIF. Keep the SIF and Apptainer caches in ephemeral storage.
- Every real collection uses the standard mini-swe runner so the task branch is checked out before the agent starts.
- Evaluate and analyze a bounded pilot before selecting full-dataset shard counts and walltimes. Start each attempt with a new dated run root; delete and recreate instead of updating it in place.